# Experiment 5: Subword Tokenization and POS Tagging

This notebook is designed to run in your **existing Conda/Jupyter environment**. It does not create a new environment and does not install packages.

Sections:
1. BPE tokenization with a pretrained GPT-2 tokenizer
2. BPE implemented from scratch
3. Pretrained T5 (SentencePiece-based) tokenization
4. Training a SentencePiece BPE tokenizer
5. POS tagging with spaCy
6. POS tagging with NLTK
7. Unique tokens, POS descriptions, and frequencies with spaCy

**Input file:** `input_sub_word_data.txt`

Keep this file in the same folder as the notebook. If it is not found, a short fallback corpus is used.

## 0. Check the current kernel and dependencies

Run this first. It checks which Python interpreter the notebook is using and verifies the main libraries without installing or changing anything.

If the imports succeed, continue with the experiment.

In [11]:
import scipy
import sklearn
import transformers
import sentencepiece
import spacy
import nltk

## Load the input corpus

In [2]:
from pathlib import Path

input_path = Path("input_sub_word_data.txt")

if input_path.exists():
    text = input_path.read_text(encoding="utf-8")
    print(f"Loaded corpus from: {input_path.resolve()}")
else:
    text = (
        "Natural language processing is a branch of artificial intelligence that "
        "enables computers to understand, interpret, and generate human language. "
        "Byte Pair Encoding is a popular subword tokenization algorithm. "
        "Transformers use attention mechanisms to identify relationships between tokens. "
        "Machine learning algorithms require numerical representations of text."
    )
    print("input_sub_word_data.txt was not found; using the fallback corpus.")

print("Character count:", len(text))
print("\nPreview:\n")
print(text[:1000])

Loaded corpus from: /Users/saiuppalapati/CL - NLP/Experiment  - 5/input_sub_word_data.txt
Character count: 6682

Preview:

Natural language processing is a branch of artificial intelligence that
enables computers to understand, interpret, and generate human language.
Natural language processing combines linguistics, computer science, and
machine learning to process large amounts of natural language data.

Tokenization is one of the fundamental steps in natural language processing.
A tokenizer divides text into smaller units called tokens. Tokens may be
words, characters, or subword units. Word-level tokenization is simple and
easy to understand, but it may have difficulty handling rare and unknown
words.

Subword-level tokenization provides an effective solution to the problem
of rare words. Instead of representing every complete word as a separate
token, a subword tokenizer represents a word using smaller meaningful units.
For example, the word playing can be represented as play and i

## 1.1 - BPE tokenization using a pretrained tokenizer

Load the pretrained GPT-2 tokenizer, split the corpus into subword tokens, and convert the tokens to numerical IDs. The tokenizer files must already be cached locally or be downloadable through your internet connection.

In [3]:
from transformers import AutoTokenizer

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

bpe_tokens = gpt2_tokenizer.tokenize(text)
bpe_token_ids = gpt2_tokenizer.convert_tokens_to_ids(bpe_tokens)

print("First 100 BPE tokens:")
print(bpe_tokens[:100])

print("\nFirst 100 token IDs:")
print(bpe_token_ids[:100])

print("\nNumber of tokens:", len(bpe_tokens))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1348 > 1024). Running this sequence through the model will result in indexing errors


First 100 BPE tokens:
['Natural', 'Ġlanguage', 'Ġprocessing', 'Ġis', 'Ġa', 'Ġbranch', 'Ġof', 'Ġartificial', 'Ġintelligence', 'Ġthat', 'Ċ', 'en', 'ables', 'Ġcomputers', 'Ġto', 'Ġunderstand', ',', 'Ġinterpret', ',', 'Ġand', 'Ġgenerate', 'Ġhuman', 'Ġlanguage', '.', 'Ċ', 'Natural', 'Ġlanguage', 'Ġprocessing', 'Ġcombines', 'Ġlingu', 'istics', ',', 'Ġcomputer', 'Ġscience', ',', 'Ġand', 'Ċ', 'machine', 'Ġlearning', 'Ġto', 'Ġprocess', 'Ġlarge', 'Ġamounts', 'Ġof', 'Ġnatural', 'Ġlanguage', 'Ġdata', '.', 'Ċ', 'Ċ', 'Token', 'ization', 'Ġis', 'Ġone', 'Ġof', 'Ġthe', 'Ġfundamental', 'Ġsteps', 'Ġin', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', '.', 'Ċ', 'A', 'Ġtoken', 'izer', 'Ġdivides', 'Ġtext', 'Ġinto', 'Ġsmaller', 'Ġunits', 'Ġcalled', 'Ġtokens', '.', 'ĠTokens', 'Ġmay', 'Ġbe', 'Ċ', 'words', ',', 'Ġcharacters', ',', 'Ġor', 'Ġsub', 'word', 'Ġunits', '.', 'ĠWord', '-', 'level', 'Ġtoken', 'ization', 'Ġis', 'Ġsimple', 'Ġand', 'Ċ', 'easy', 'Ġto', 'Ġunderstand']

First 100 token IDs:
[35364, 3303, 7587, 318, 2

## 1.2 - BPE tokenization without a pretrained tokenizer

This educational implementation starts with characters plus an end-of-word marker, then repeatedly merges the most frequent adjacent pair. It learns merges from the current corpus and assigns IDs to the resulting symbols.

Note: this demonstrates BPE training; it is not a complete encoder for arbitrary unseen words.

In [4]:
import re
from collections import Counter

# Extract lowercase alphabetic words from the corpus.
words = re.findall(r"\b[a-z]+\b", text.lower())

# Represent each word as characters followed by an end-of-word marker.
bpe_vocab = Counter()
for word in words:
    bpe_vocab[" ".join(list(word)) + " </w>"] += 1

num_merges = 20
learned_merges = []

for step in range(num_merges):
    pair_counts = Counter()

    # Count adjacent symbol pairs, weighted by each word's frequency.
    for word_representation, frequency in bpe_vocab.items():
        symbols = word_representation.split()
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += frequency

    if not pair_counts:
        break

    best_pair = pair_counts.most_common(1)[0][0]
    learned_merges.append(best_pair)

    # Apply the selected merge to every vocabulary entry.
    updated_vocab = Counter()
    for word_representation, frequency in bpe_vocab.items():
        symbols = word_representation.split()
        merged_symbols = []
        i = 0

        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and (symbols[i], symbols[i + 1]) == best_pair
            ):
                merged_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                merged_symbols.append(symbols[i])
                i += 1

        updated_vocab[" ".join(merged_symbols)] += frequency

    bpe_vocab = updated_vocab
    print(f"Merge {step + 1:02d}: {best_pair[0]} + {best_pair[1]}")

# Collect the symbols remaining after training.
learned_symbols = set()
for word_representation in bpe_vocab:
    learned_symbols.update(word_representation.split())

# Assign deterministic numerical IDs to the learned symbols.
bpe_token_to_id = {
    symbol: idx for idx, symbol in enumerate(sorted(learned_symbols))
}

print("\nLearned vocabulary entries (first 30):")
for item, frequency in list(bpe_vocab.items())[:30]:
    print(f"{item}: {frequency}")

print("\nLearned symbol vocabulary size:", len(bpe_token_to_id))
print("\nExample token IDs (first 30 symbols):")
print(dict(list(bpe_token_to_id.items())[:30]))

Merge 01: e + </w>
Merge 02: s + </w>
Merge 03: e + n
Merge 04: i + n
Merge 05: d + </w>
Merge 06: e + r
Merge 07: o + r
Merge 08: a + n
Merge 09: t + h
Merge 10: o + n
Merge 11: a + t
Merge 12: a + r
Merge 13: r + e
Merge 14: a + l
Merge 15: in + g
Merge 16: t + o
Merge 17: y + </w>
Merge 18: ing + </w>
Merge 19: en + t
Merge 20: i + on

Learned vocabulary entries (first 30):
n at u r al </w>: 6
l an g u a g e</w>: 12
p r o c e s s ing</w>: 7
i s</w>: 14
a </w>: 28
b r an c h </w>: 1
o f </w>: 22
ar t i f i c i al </w>: 1
in t e l l i g en c e</w>: 1
th at </w>: 4
en a b l e s</w>: 1
c o m p u t er s</w>: 1
to </w>: 15
u n d er s t an d</w>: 4
in t er p re t </w>: 1
an d</w>: 27
g en er at e</w>: 1
h u m an </w>: 1
c o m b in e s</w>: 1
l ing u i s t i c s</w>: 1
c o m p u t er </w>: 1
s c i en c e</w>: 1
m a c h in e</w>: 4
l e ar n ing</w>: 6
p r o c e s s</w>: 3
l ar g e</w>: 4
a m o u n t s</w>: 1
d at a </w>: 4
to k en i z at ion </w>: 8
on e</w>: 2

Learned symbol vocabulary siz

## 2.1 - Pretrained SentencePiece-based tokenization

T5 uses a SentencePiece-based tokenizer. This section loads the pretrained `t5-small` tokenizer and displays its subword tokens and IDs.

In [5]:
from transformers import AutoTokenizer

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")

sentencepiece_tokens = t5_tokenizer.tokenize(text)
sentencepiece_token_ids = t5_tokenizer.convert_tokens_to_ids(sentencepiece_tokens)

print("First 100 tokens:")
print(sentencepiece_tokens[:100])

print("\nFirst 100 token IDs:")
print(sentencepiece_token_ids[:100])

print("\nNumber of tokens:", len(sentencepiece_tokens))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1319 > 512). Running this sequence through the model will result in indexing errors


First 100 tokens:
['▁Natural', '▁language', '▁processing', '▁is', '▁', 'a', '▁branch', '▁of', '▁artificial', '▁intelligence', '▁that', '▁', 'enables', '▁computers', '▁to', '▁understand', ',', '▁interpret', ',', '▁and', '▁generate', '▁human', '▁language', '.', '▁Natural', '▁language', '▁processing', '▁', 'combines', '▁', 'linguistic', 's', ',', '▁computer', '▁science', ',', '▁and', '▁machine', '▁learning', '▁to', '▁process', '▁large', '▁amounts', '▁of', '▁natural', '▁language', '▁data', '.', '▁To', 'ken', 'ization', '▁is', '▁one', '▁of', '▁the', '▁fundamental', '▁steps', '▁in', '▁natural', '▁language', '▁processing', '.', '▁A', '▁token', 'izer', '▁divide', 's', '▁text', '▁into', '▁smaller', '▁units', '▁called', '▁token', 's', '.', '▁To', 'ken', 's', '▁may', '▁be', '▁words', ',', '▁characters', ',', '▁or', '▁sub', 'word', '▁units', '.', '▁Word', '-', 'level', '▁token', 'ization', '▁is', '▁simple', '▁and', '▁easy', '▁to', '▁understand']

First 100 token IDs:
[6869, 1612, 3026, 19, 3, 9, 6

## 2.2 - Train a SentencePiece BPE tokenizer

This section trains a new SentencePiece tokenizer on the corpus. It writes `my_bpe.model` and `my_bpe.vocab` in the notebook's working directory. No packages are installed by this cell.

In [6]:
import sentencepiece as spm

training_file = Path("input_sub_word_data.txt")
if not training_file.exists():
    training_file.write_text(text, encoding="utf-8")

spm.SentencePieceTrainer.train(
    input=str(training_file),
    model_prefix="my_bpe",
    vocab_size=200,
    model_type="bpe",
    character_coverage=1.0,
    hard_vocab_limit=False
)

sp = spm.SentencePieceProcessor(model_file="my_bpe.model")

sp_tokens = sp.encode(text, out_type=str)
sp_token_ids = sp.encode(text, out_type=int)

print("First 100 SentencePiece tokens:")
print(sp_tokens[:100])

print("\nFirst 100 token IDs:")
print(sp_token_ids[:100])

print("\nNumber of tokens:", len(sp_tokens))
print("Actual vocabulary size:", sp.get_piece_size())
print("\nSaved model: my_bpe.model")
print("Saved vocabulary: my_bpe.vocab")

First 100 SentencePiece tokens:
['▁', 'N', 'at', 'ural', '▁language', '▁pro', 'cessing', '▁is', '▁a', '▁b', 'r', 'an', 'c', 'h', '▁of', '▁a', 'r', 't', 'if', 'ic', 'i', 'al', '▁in', 't', 'el', 'l', 'i', 'g', 'ence', '▁th', 'at', '▁', 'en', 'ab', 'l', 'es', '▁com', 'pu', 'ter', 's', '▁to', '▁u', 'nd', 'er', 'st', 'an', 'd', ',', '▁in', 'ter', 'p', 're', 't', ',', '▁and', '▁', 'g', 'en', 'er', 'at', 'e', '▁h', 'u', 'm', 'an', '▁language', '.', '▁', 'N', 'at', 'ural', '▁language', '▁pro', 'cessing', '▁com', 'b', 'in', 'es', '▁l', 'ing', 'u', 'is', 'tic', 's', ',', '▁com', 'pu', 'ter', '▁s', 'c', 'i', 'ence', ',', '▁and', '▁m', 'ac', 'h', 'in', 'e', '▁learn']

First 100 token IDs:
[154, 191, 12, 129, 118, 71, 130, 84, 7, 32, 159, 15, 164, 168, 58, 7, 159, 158, 75, 46, 162, 21, 50, 158, 61, 163, 162, 170, 153, 17, 12, 154, 4, 73, 163, 11, 112, 76, 55, 161, 22, 30, 34, 6, 78, 15, 166, 173, 50, 55, 169, 18, 158, 173, 45, 154, 170, 4, 6, 12, 155, 74, 165, 167, 15, 118, 175, 154, 191, 12, 129, 

I0000 00:00:1791989364.552586  841464 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: input_sub_word_data.txt
  input_format: 
  model_prefix: my_bpe
  model_type: BPE
  vocab_size: 200
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_dif

## 3.1 - POS tagging using spaCy

POS tagging assigns grammatical categories such as nouns, verbs, adjectives, and adverbs. This cell expects the spaCy English model `en_core_web_sm` to be available in the current environment; it does not install it automatically.

In [7]:
import spacy

nlp = spacy.load("en_core_web_sm")

sample_text = (
    "Natural language processing enables computers to understand human language. "
    "Machine learning improves language models."
)

doc = nlp(sample_text)

print(f"{'Token':<18}{'POS Tag':<12}Description")
print("-" * 55)

for token in doc:
    print(
        f"{token.text:<18}{token.pos_:<12}"
        f"{spacy.explain(token.pos_) or 'Unknown'}"
    )

Token             POS Tag     Description
-------------------------------------------------------
Natural           ADJ         adjective
language          NOUN        noun
processing        NOUN        noun
enables           VERB        verb
computers         NOUN        noun
to                PART        particle
understand        VERB        verb
human             ADJ         adjective
language          NOUN        noun
.                 PUNCT       punctuation
Machine           NOUN        noun
learning          NOUN        noun
improves          VERB        verb
language          NOUN        noun
models            NOUN        noun
.                 PUNCT       punctuation


## 3.2 - POS tagging using NLTK

NLTK's `pos_tag` generally uses Penn Treebank tags, such as `NN` for a singular noun and `VBG` for a gerund or present participle. The required NLTK data resources must already be present or be downloadable.

In [8]:
import nltk

# Download only NLTK data resources if they are missing; this does not install packages.
for resource in [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]:
    try:
        nltk.data.find(
            "tokenizers/punkt" if resource == "punkt" else
            "tokenizers/punkt_tab" if resource == "punkt_tab" else
            "taggers/averaged_perceptron_tagger" if resource == "averaged_perceptron_tagger" else
            "taggers/averaged_perceptron_tagger_eng"
        )
    except LookupError:
        nltk.download(resource, quiet=True)

sample_text = (
    "Natural language processing enables computers to understand human language. "
    "Machine learning improves language models."
)

tokens = nltk.word_tokenize(sample_text)
tagged_tokens = nltk.pos_tag(tokens)

print(f"{'Token':<20}{'POS Tag'}")
print("-" * 32)

for token, tag in tagged_tokens:
    print(f"{token:<20}{tag}")

Token               POS Tag
--------------------------------
Natural             JJ
language            NN
processing          NN
enables             VBZ
computers           NNS
to                  TO
understand          VB
human               JJ
language            NN
.                   .
Machine             NNP
learning            VBG
improves            NNS
language            NN
models              NNS
.                   .


## 4 - Unique tokens, POS descriptions, and frequencies using spaCy

This section ignores punctuation and whitespace, groups tokens without regard to letter case, and reports each unique token's POS tag, description, and frequency.

In [9]:
from collections import Counter

doc = nlp(text)

token_frequency = Counter(
    token.text.lower()
    for token in doc
    if not token.is_space and not token.is_punct
)

token_information = {}

for token in doc:
    if token.is_space or token.is_punct:
        continue

    word = token.text.lower()
    if word not in token_information:
        token_information[word] = (
            token.pos_,
            spacy.explain(token.pos_) or "Unknown"
        )

print(f"{'Token':<24}{'POS Tag':<12}{'Description':<28}{'Frequency'}")
print("-" * 78)

for word in sorted(token_information):
    pos_tag, description = token_information[word]
    print(
        f"{word:<24}{pos_tag:<12}{description:<28}"
        f"{token_frequency[word]}"
    )

print("\nNumber of unique tokens:", len(token_information))

Token                   POS Tag     Description                 Frequency
------------------------------------------------------------------------------
a                       DET         determiner                  28
able                    ADJ         adjective                   1
above                   ADJ         adjective                   1
according               VERB        verb                        1
additional              ADJ         adjective                   1
adjacent                ADJ         adjective                   2
after                   ADP         adposition                  2
algorithm               PROPN       proper noun                 5
algorithms              NOUN        noun                        1
all                     PRON        pronoun                     1
allowing                VERB        verb                        1
allows                  VERB        verb                        2
also                    ADV         adverb            

## Conclusion

- **BPE** learns merges of frequent adjacent symbols to build subword units.
- **SentencePiece** can train a subword tokenizer directly from a text corpus.
- **Token IDs** represent tokens numerically for model input.
- **POS tagging** assigns grammatical labels to tokens.
- **Frequency analysis** counts how often each token occurs.

### Running notes
- Use the kernel connected to your existing environment (`tf`); this notebook does not create another environment or install packages.
- If the first diagnostic cell fails, copy the specific error message. If it passes, continue through the notebook.
- Pretrained tokenizer files may need to be downloaded the first time if they are not cached.